In [10]:
!uv pip install -e ../../hedyPET

Using Python 3.11.11 environment at: /homes/hinge/Projects/hedypet-streamlit/.venv
Resolved 72 packages in 738ms                                        
   Building hedypet @ file:///homes/hinge/Projects/hedyPET             
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
   Building hedypet @ file:///homes/hinge/Projects/hedyPET     
      Built hedypet @ file:///homes/hinge/Projects/hedyPET     
Prepared 1 package in 819ms                                              
Uninstalled 1 package in 4ms
Installed 1 package in 19ms file:///homes/hinge/Projects/hed
 ~ hedypet==0.1.0 (from file:///homes/hinge/Projects/hedyPET)


In [3]:
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from matplotlib import pyplot as plt
from parse import parse
import numpy as np
import os 
import pandas as pd 
from hedypet.utils import get_norm_consts
from hedypet.utils import load_splits 


df = pd.read_pickle("/homes/hinge/Projects/hedyPET/src/hedypet/analysis/acstatPSF_means.pkl")
RAW = Path("/depict/data/hedit/raw/")
DERIVATIVES = Path("/depict/data/hedit/derivatives/pipeline-bodydyn/")

def filter_and_combine_regions(df,regions):
    group_cols = ["sub","erosion","task"]
    results = []

    for k, v in regions.items():
        mask = df.region.isin(v) if isinstance(v, list) else df.region == v
        filtered = df[mask]
        
        if isinstance(v, list):
            result = filtered.groupby(group_cols).apply(
                lambda x: pd.Series({
                    'mu': (x.mu * x.n).sum() / x.n.sum(),
                    'n': x.n.sum()
                })
            ).reset_index()
        else:
            result = filtered
        
        result['region'] = k
        results.append(result)

    return pd.concat(results, ignore_index=True)

regions = {
    "Spleen":"spleen",
    "Kidneys": ["kidney_right","kidney_left"],
    "Aorta": "aorta",
    "Liver":"liver",
    "Stomach":"stomach",
    "Lungs": ['lung_upper_lobe_left', 'lung_lower_lobe_left','lung_upper_lobe_right', 'lung_middle_lobe_right','lung_lower_lobe_right'],
    'Colon':"colon",
    'Subcutaneous fat':"subcutaneous_fat",
    "Skeletal muscle":"skeletal_muscle",
    'Visceral fat':"visceral_fat",
    "Gray matter":['Left-Cerebral-Cortex', 'Right-Cerebral-Cortex'], 
    "White matter": ['Right-Cerebral-White-Matter' , 'Left-Cerebral-White-Matter'],
    "Brain":"brain",
    "Esophagus": "esophagus",
    "Trachea": "trachea",
    "Small intestine": ["small_bowel","duodenum"],
    "Skull": "skull",
    "Ribs": [f"rib_left_{i}" for i in range(1,13)] + [f"rib_right_{i}" for i in range(1,13)],
    "Urinary bladder": "Urinary bladder",
    "Gallbladder": "gallbladder",
    "Pancreas": "pancreas",
    "Heart": "heart"
}
df = filter_and_combine_regions(df,regions)

/tmp/ipykernel_3731913/450538202.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['region'] = k
/tmp/ipykernel_3731913/450538202.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = filtered.groupby(group_cols).apply(
/tmp/ipykernel_3731913/450538202.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [8]:
import pandas as pd
df_norm = pd.DataFrame([get_norm_consts(sub) for sub in load_splits()["all"]])
df_norm.to_pickle("norm2.pkl")